# 🔬 01 — Data Preprocessing, Quality Audit & Exploratory Data Analysis

The data foundation of **CryptoGraph-Secure**: 20 model coins over 10 years (2016-09-28 → 2026-09-27), from Yahoo Finance,
cross-checked against Coin Metrics. LUNC (Terra) and FTT (FTX) are kept as event coins because those crises started there.
*Why not CoinGecko?* Its free plan only serves the last 365 days of history.

1. **Quality audit** — what the checks found, and how reliable each period is.
2. **Staggered launches & active masking** — when each coin's data starts and from which source.
3. **Prices and crises.**
4. **Stationarity (ADF)** — prices vs returns.
5. **Fat tails** — Jarque-Bera and kurtosis.
6. **Autocorrelation** — direction vs volatility.
7. **Shock & contagion labels.**
8. **Train-only robust scaling.**
9. **Splits, embargo & distribution shift.**
10. **Correlation** — why a graph model.
11. **Dynamic graph** — edges over time and the threshold.
12. **Tensor & DataLoader check.**

**Part II (13–18)** adds the hourly Binance data (30 coins) that the models train on: coverage, seasonality, lead-lag, the Epps effect and graph threshold, hourly labels, and the hourly tensors.

All Part I statistics use **raw daily log returns from the cleaned prices** (`data/interim/master.parquet`), not the scaled model inputs.

In [1]:
import sys, json
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"

from cryptograph.data import load_processed_dataset

data = load_processed_dataset("../data/processed/crypto_graph_dataset.pt")   # graph-model tensors (20 coins)
panel = torch.load("../data/processed/v1/panel.pt", weights_only=False)        # features/targets, all coins
master = pd.read_parquet("../data/interim/master.parquet")                     # cleaned daily prices
events = pd.read_csv("../data/external/events.csv", parse_dates=["date"])
audit = pd.read_csv("../reports/universe_audit.csv")
quality = pd.read_csv("../reports/data_quality_report.csv").set_index("coin")
balance = pd.read_csv("../reports/label_balance.csv")
splits = json.loads(Path("../data/processed/v1/splits.json").read_text())["splits"]

asset_names = data["asset_names"]
timestamps = pd.to_datetime(data["timestamps"])
close = master["close"].unstack("coin")
log_returns_df = np.log(close).diff()                 # raw daily log returns
r20 = log_returns_df[asset_names]
panel_a = [c for c, a in zip(panel["coins"], panel["panel_A"]) if a]
day_split = np.array(panel["day_split"])

def add_events(fig, size=9):
    for _, e in events.iterrows():
        fig.add_vline(x=e["date"], line_width=1.2, line_dash="dash", line_color="#ff5252", opacity=0.7)
        fig.add_annotation(x=e["date"], y=1, yref="paper", text=e["name"].split("/")[0].strip()[:28], showarrow=False,
                           textangle=-90, xanchor="right", yanchor="top", font=dict(color="#ff5252", size=size))
    return fig

print(f"📊 {len(timestamps)} days × {len(asset_names)} model coins × {len(data['feature_names'])} features "
      f"({timestamps[0].date()} → {timestamps[-1].date()}); event coins: LUNC, FTT")
print("🪙", ", ".join(asset_names))

📊 3652 days × 20 model coins × 12 features (2016-09-28 → 2026-09-27); event coins: LUNC, FTT
🪙 BTC, ETH, BNB, XRP, SOL, TRX, ZEC, DOGE, LINK, XMR, WBT, ADA, LEO, XLM, NEAR, BCH, UNI, LTC, AVAX, HBAR


*What this shows:* Loads the pipeline outputs: the graph-model tensors for the 20 model coins, the feature/target panel, the cleaned daily
prices and the verified crisis dates (`data/external/events.csv`, each with a source URL). The 20 coins are today's
largest non-stable coins that have enough history and liquidity; returns are computed from the cleaned prices.

## 1. Data Quality Audit

Every coin-day passes through the same checks: duplicate dates, missing days, zero or negative prices, high < low,
open/close outside the day's range, zero volume, volume spikes, stale prices, and very large moves (>40%) verified against
Coin Metrics or a known event. Days before a coin existed are **not** counted as missing: they are simply inactive.

In [2]:
q = quality.loc["ALL"]
audit_table = pd.DataFrame({
    "Check": ["Active coin-days", "Bad prices corrected", "Very large moves (>40%)", "  … confirmed by 2nd source / event",
              "  … no 2nd source (kept, flagged)", "Missing days (left inactive)", "Gaps forward-filled",
              "High < low", "Zero or negative prices", "Duplicate dates"],
    "Count": [q["active_days"], q["bad_tick_corrected"], q["large_return"], q["large_return_confirmed"], q["large_return_unverified"],
              q["missing_day"], q["filled"], q["high_lt_low"], q["nonpositive_price"], q["duplicate_date"]]}).astype({"Count": int})
display(audit_table)

rec = pd.read_parquet("../data/interim/reconciled.parquet").reset_index()
by_year = (rec[rec["xsrc_flag"]].groupby(rec["date"].dt.year).size()
           / rec[rec["xsrc_rel_diff"].notna()].groupby(rec["date"].dt.year).size()).fillna(0)
fig = px.bar(x=by_year.index.astype(str), y=by_year.values, text=[f"{v:.1%}" for v in by_year.values],
             labels={"x": "year", "y": "share of days"}, title="Days where Yahoo and Coin Metrics disagree by more than 2%")
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()

,Check,Count
0,Active coin-days,90643
1,Bad prices corrected,28
2,Very large moves (>40%),156
3,… confirmed by 2nd source / event,123
4,"… no 2nd source (kept, flagged)",33
5,Missing days (left inactive),69
6,Gaps forward-filled,0
7,High < low,0
8,Zero or negative prices,0
9,Duplicate dates,0


*What this shows:* Nothing structurally wrong was found: no high < low, no zero prices, no duplicate dates, and no gap needed filling.
28 bad prices were corrected, 123 of the 156 very large moves were confirmed by Coin Metrics or a catalogued crisis, and the
only missing stretch (AVAX, 69 days) was left empty and inactive. The bar chart shows the two sources disagree on 23% of
days in 2017 but only ~0.2% from 2022, so results that depend on 2017–18 should be read with care.

## 2. Staggered Coin Launches & Active Masking

Older coins (BTC, LTC, ETH, XRP) have data from 2016; newer ones (SOL, AVAX, NEAR, …) start in 2019–2022. Instead of
inventing earlier prices, every coin carries an **active mask** $m_i(t) \in \{0, 1\}$ that the models use.
Most coins have full Yahoo data only from **2017-11-09**; before that there are closing prices from Coin Metrics.

In [3]:
src = master["close_source"].unstack("coin")
rows = []
for coin in asset_names + ["LUNC", "FTT"]:
    for label, key in (("Coin Metrics (close only)", "coinmetrics"), ("Yahoo (full OHLCV)", "yahoo")):
        d = src.index[src[coin] == key]
        if len(d):
            rows.append({"coin": coin, "source": label, "start": d.min(),
                         "end": (timestamps[-1] if key == "yahoo" else d.max()) + pd.Timedelta(days=1)})
fig = px.timeline(pd.DataFrame(rows), x_start="start", x_end="end", y="coin", color="source",
                  category_orders={"coin": asset_names + ["LUNC", "FTT"]},
                  color_discrete_map={"Coin Metrics (close only)": "#80cbc4", "Yahoo (full OHLCV)": "#1e88e5"},
                  title="Data coverage per coin and source")
fig.add_vline(x=pd.Timestamp("2017-11-09"), line_dash="dash", line_color="orange")
fig.update_layout(height=600, xaxis_range=[timestamps[0], timestamps[-1]], legend=dict(orientation="h", y=-0.08))
fig.show()

active_df = pd.DataFrame(data["active_mask"].numpy(), index=timestamps, columns=asset_names)
debut_df = pd.DataFrame({"first_active": [active_df.index[active_df[c]].min().date() for c in asset_names],
                         "active_days": active_df.sum().values,
                         "coverage_%": (active_df.mean().values * 100).round(1)}, index=asset_names).sort_values("first_active")
display(debut_df.T)

,BTC,LTC,XLM,XMR,ETH,XRP,ZEC,BNB,BCH,LINK,TRX,DOGE,ADA,LEO,HBAR,SOL,AVAX,UNI,NEAR,WBT
first_active,2016-09-28,2016-09-28,2016-09-28,2016-09-28,2016-09-28,2016-09-28,2016-10-29,2017-07-15,2017-08-01,2017-09-29,2017-11-09,2017-11-09,2017-11-09,2019-05-21,2019-09-17,2020-04-10,2020-07-13,2020-09-18,2020-10-14,2022-08-26
active_days,3652,3652,3652,3652,3652,3652,3621,3362,3345,3286,3245,3245,3245,2687,2568,2362,2199,2201,2175,1494
coverage_%,100.0,100.0,100.0,100.0,100.0,100.0,99.2,92.1,91.6,90.0,88.9,88.9,88.9,73.6,70.3,64.7,60.2,60.3,59.6,40.9


*What this shows:* Each bar is one coin: **dark blue** = full Yahoo data (open, high, low, close, volume), **teal** = only a closing price from
Coin Metrics; the orange line is 2017-11-09. So the dataset is 10 years of closing prices and about 9 years of full OHLCV.
The table gives each coin's first active day and coverage; newer coins simply start later and are masked inactive before that.

## 3. 10-Year Price Growth & Historical Crises

Growth of each coin from its first available price (log scale), with the verified crisis dates.

In [4]:
selected = ["BTC", "ETH", "SOL", "BNB", "DOGE", "ADA", "LINK", "AVAX"]
growth = close[selected].apply(lambda s: s / s.dropna().iloc[0])
fig = go.Figure([go.Scatter(x=growth.index, y=growth[a], mode="lines", name=a, line=dict(width=1.8)) for a in selected])
add_events(fig)
fig.update_layout(title="10-Year Growth Factor (Log Scale) with Systemic Crises", yaxis_type="log",
                  yaxis_title="Cumulative Growth Factor", height=580,
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig.show()

*What this shows:* Prices move across several orders of magnitude and trend over time, and every major crisis (red lines) hits all coins at
the same time. Trending prices can't be fed to a model directly; the next section tests this formally.

## 4. Stationarity (Augmented Dickey-Fuller)

A series with a unit root (non-stationary) makes regressions spurious. We run the ADF test on **log prices** and on **daily
log returns** for each coin; a p-value below 0.01 means the series is stationary.

In [5]:
adf_rows = []
for a in asset_names:
    s_p = adfuller(np.log(close[a].dropna()), autolag="AIC")
    s_r = adfuller(r20[a].dropna(), autolag="AIC")
    adf_rows.append({"Asset": a, "ADF stat (log price)": round(s_p[0], 2), "p (log price)": round(s_p[1], 3),
                     "ADF stat (return)": round(s_r[0], 2), "p (return)": f"{s_r[1]:.1e}",
                     "Price stationary": "yes" if s_p[1] < 0.05 else "no", "Return stationary": "yes" if s_r[1] < 0.01 else "no"})
adf_df = pd.DataFrame(adf_rows).set_index("Asset")
display(adf_df.T)
print(f"Log prices non-stationary (p > 0.05) for {(adf_df['Price stationary'] == 'no').sum()}/20 coins; "
      f"returns stationary (p < 0.01) for {(adf_df['Return stationary'] == 'yes').sum()}/20")

Asset,BTC,ETH,BNB,XRP,SOL,TRX,ZEC,DOGE,LINK,XMR,WBT,ADA,LEO,XLM,NEAR,BCH,UNI,LTC,AVAX,HBAR
ADF stat (log price),-2.58,-2.76,-2.9,-3.26,-2.82,-3.47,-2.2,-1.55,-1.73,-2.66,-0.38,-2.15,-0.52,-3.23,-2.21,-2.71,-2.05,-3.5,-2.44,-2.06
p (log price),0.097,0.064,0.046,0.017,0.056,0.009,0.205,0.509,0.415,0.081,0.913,0.226,0.887,0.018,0.201,0.073,0.265,0.008,0.13,0.262
ADF stat (return),-41.86,-17.8,-12.2,-12.82,-10.18,-9.44,-40.55,-9.7,-59.88,-67.91,-8.18,-10.5,-11.44,-18.04,-46.9,-39.85,-22.09,-21.49,-24.99,-24.42
p (return),0.0e+00,3.2e-30,1.3e-22,6.3e-24,6.8e-18,4.8e-16,0.0e+00,1.1e-16,0.0e+00,0.0e+00,8.2e-13,1.1e-18,6.3e-21,2.7e-30,0.0e+00,0.0e+00,0.0e+00,0.0e+00,0.0e+00,0.0e+00
Price stationary,no,no,yes,yes,no,yes,no,no,no,no,no,no,no,yes,no,no,no,yes,no,no
Return stationary,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes,yes


Log prices non-stationary (p > 0.05) for 15/20 coins; returns stationary (p < 0.01) for 20/20


*What this shows:* Log prices are non-stationary for 15 of the 20 coins (the test cannot reject a unit root), while daily returns are
stationary for all 20. That is why every model input is built from **returns and volatility**, never raw prices.

## 5. Fat Tails & Extreme Leptokurtosis (Jarque-Bera & Kurtosis)

Crypto returns are not normally distributed. Below, Bitcoin's return density is compared with a normal distribution on a
**logarithmic** probability axis, and the Jarque-Bera test is run for every coin.

In [6]:
kurt = r20.kurt()
jb_rejected = sum(stats.jarque_bera(r20[a].dropna()).pvalue < 0.01 for a in asset_names)
print(f"📊 Median excess kurtosis: {kurt.median():.2f} (normal distribution = 0)")
print(f"🚨 Normality rejected (Jarque-Bera p < 0.01): {jb_rejected} / {len(asset_names)} coins")

btc = r20["BTC"].dropna() * 100
grid = np.linspace(btc.min(), btc.max(), 300)
fig = go.Figure([
    go.Histogram(x=btc, histnorm="probability density", nbinsx=150, name="BTC daily log returns",
                 marker=dict(color="#2979ff", opacity=0.75)),
    go.Scatter(x=grid, y=stats.norm.pdf(grid, btc.mean(), btc.std()), mode="lines", name="Normal distribution",
               line=dict(color="#ff1744", width=2.5))])
fig.update_layout(title="Bitcoin Daily Return Density vs Normal (log scale reveals the fat tails)",
                  xaxis_title="Daily log return (%)", yaxis_type="log", yaxis_title="Probability density (log)",
                  height=450, legend=dict(x=0.02, y=0.98))
fig.show()

📊 Median excess kurtosis: 18.24 (normal distribution = 0)
🚨 Normality rejected (Jarque-Bera p < 0.01): 20 / 20 coins


*What this shows:* On the log scale the red normal curve vanishes beyond about ±10%, but BTC has many days out there (worst: −37% on
2020-03-12). Median excess kurtosis is about 18 and normality is rejected for all 20 coins. A squared-error loss would be
dominated by these few days, so the models use a **Huber loss** and inputs are **clipped** at ±5 after scaling.

## 6. Autocorrelation: Direction vs Volatility

Autocorrelation over 10 daily lags for raw returns $r_t$ (direction) and squared returns $r_t^2$ (volatility).

In [7]:
train_r = r20.loc[timestamps[day_split == "train"]]
same_sign = (np.sign(train_r.shift(1)) == np.sign(train_r)).where(train_r.notna() & train_r.shift(1).notna()).stack().mean()
acf_r = pd.DataFrame({a: acf(r20[a].dropna(), nlags=10, fft=True)[1:] for a in asset_names}).mean(axis=1)
acf_v = pd.DataFrame({a: acf(r20[a].dropna() ** 2, nlags=10, fft=True)[1:] for a in asset_names}).mean(axis=1)
lags = list(range(1, 11))
fig = go.Figure([go.Bar(x=lags, y=acf_r.values, name="Returns rₜ (direction)", marker=dict(color="#00e5ff")),
                 go.Bar(x=lags, y=acf_v.values, name="Squared returns rₜ² (volatility)", marker=dict(color="#ff9100"))])
fig.update_layout(barmode="group", title="Autocorrelation over 10 lags, average of 20 coins",
                  xaxis_title="Lag (days)", yaxis_title="Autocorrelation", height=420, legend=dict(x=0.6, y=0.95))
fig.show()
print(f"📈 Lag-1 autocorrelation — direction: {acf_r.iloc[0]:+.3f}, volatility: {acf_v.iloc[0]:+.3f}; "
      f"P(tomorrow's sign = today's) on train = {same_sign:.2f}")

📈 Lag-1 autocorrelation — direction: -0.024, volatility: +0.184; P(tomorrow's sign = today's) on train = 0.47


*What this shows:* The blue bars (direction) are about zero at every lag, and the same sign repeats only 47% of the time: yesterday's
direction says nothing about today's. The orange bars (volatility) are clearly positive (about 0.18 at lag 1): calm days
follow calm days and turbulent days follow turbulent days. So the models predict **volatility and shocks**, not direction.

## 7. Shock & Contagion Labels

- **Shock:** a day where |z| > 2.5, with z = return ÷ trailing 30-day volatility measured **up to the day before**.
- **Contagion:** a shock at coin *i* followed by a shock at coin *j* within *k* = 1–3 days (a time lag, not same-day co-occurrence).

In [8]:
sh = balance[balance["label"].str.startswith("shock_next")]
fig = px.bar(sh, x="split", y="rate", text=[f"{v:.1%}" for v in sh["rate"]], title="Shock label: share of positive days per split")
fig.update_layout(height=340, yaxis_tickformat=".0%")
fig.show()

ct = balance[balance["label"].str.startswith("contagion") & (balance["split"] == "train")].copy()
ct["k"] = "within " + ct["label"].str[-1] + " day(s)"
long = pd.concat([ct.assign(case="any coin pair", value=ct["rate"]),
                  ct.assign(case="after coin i shocks", value=ct["rate_given_source_shock"])])
fig = px.bar(long, x="k", y="value", color="case", barmode="group", text=[f"{v:.1%}" for v in long["value"]],
             title="Contagion (train): chance that coin j shocks, overall vs after a shock at coin i")
fig.update_layout(height=380, yaxis_tickformat=".0%", xaxis_title=None)
fig.show()

*What this shows:* Shocks are positive on about 4% of days in every split, so a model that always says "no shock" would be 96% accurate and
useless: we report **AUPRC** and use **weighted or focal loss**. The second chart is the contagion evidence: for a random
coin pair a follow-on shock is rare (under 1%), but right after coin *i* shocks, coin *j* shocks within 1–3 days 6–14% of
the time. That jump is what the graph model has to learn.

## 8. Train-Only Robust Feature Scaling (No Leakage)

Features are scaled per coin with the median and inter-quartile range, **fitted on training dates only** and then applied
to validation and test.

In [9]:
fi = panel["feature_names"].index("log_volume")
rows = []
for coin in ["BTC", "ETH", "DOGE", "LINK", "HBAR"]:
    j = panel["coins"].index(coin)
    ok = panel["X_mask"][:, j, fi].numpy()
    for label, arr in (("raw log volume", panel["X_raw"]), ("after per-coin scaling", panel["X"])):
        rows.append(pd.DataFrame({"value": arr[:, j, fi].numpy()[ok][::5], "coin": coin, "version": label}))
fig = px.box(pd.concat(rows), x="coin", y="value", color="coin", facet_col="version", points=False,
             title="Log volume before and after scaling")
fig.update_yaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(height=420, showlegend=False)
fig.show()

*What this shows:* Left: raw log volume. BTC and ETH trade far more than HBAR or LINK, so the boxes sit at very different levels. Right:
after per-coin scaling all coins are centred on 0 with similar spread. The network sees "unusually high volume for this
coin" instead of learning which coin is which, and no validation/test information enters the scaler.

## 9. Splits, 30-Day Embargo & Distribution Shift

Fixed dates from the config, chosen so every split contains a major shock: **train** 2016-09-28 → 2022-06-30 (2018 crash,
COVID, Terra), **validation** 2022-07-01 → 2023-12-31 (FTX), **test** 2024-01-01 → 2026-09-27 (Oct-2025 liquidations).
The first 30 days of validation and test are an embargo, and any sample whose input or target window crosses a boundary is dropped.

In [10]:
display(pd.DataFrame(splits).T[["start", "end", "embargo_end", "first_sample_date", "last_sample_date"]])
rv30 = r20.rolling(30).std().median(axis=1) * np.sqrt(365) * 100
fig = px.line(x=rv30.index, y=rv30.values, labels={"x": "Date", "y": "Annualised volatility (%)"},
              title="Cross-market 30-day volatility (median of 20 coins) with train / validation / test")
for name, col in (("train", "rgba(41,121,255,0.18)"), ("val", "rgba(255,145,0,0.18)"), ("test", "rgba(0,230,118,0.18)")):
    fig.add_vrect(x0=splits[name]["start"], x1=splits[name]["end"], fillcolor=col, line_width=0,
                  annotation_text=name, annotation_position="top left")
fig.update_layout(height=420)
fig.show()
shock_rate = balance[balance["label"].str.startswith("shock_next")].set_index("split")["rate"]
display(pd.DataFrame({"mean volatility %": [rv30[day_split == s].mean() for s in ("train", "val", "test")],
                      "shock rate": [shock_rate[s] for s in ("train", "val", "test")]}, index=["train", "val", "test"])
        .style.format({"mean volatility %": "{:.0f}", "shock rate": "{:.1%}"}))

,start,end,embargo_end,first_sample_date,last_sample_date
train,2016-09-28,2022-06-30,None,2016-10-27,2022-05-31
val,2022-07-01,2023-12-31,2022-07-30,2022-07-31,2023-12-01
test,2024-01-01,2026-09-27,2024-01-30,2024-01-31,2026-08-28


,mean volatility %,shock rate
train,107,3.9%
val,63,4.1%
test,72,3.9%


*What this shows:* The table gives the split dates, the end of each embargo, and the first and last usable sample dates. The chart shows
training (2016–2022) is far more volatile than validation and test (about 40% calmer), yet the shock rate stays about 4%
in every split. Raw volatility levels shift but standardised targets don't, which is why the targets are scale-free.

## 10. Cross-Asset Correlation — Why a Graph Model

Average return correlation between coins, overall and over time (60-day rolling window) with the crisis dates.

In [11]:
rA = log_returns_df.loc["2017-11-09":, panel_a].dropna()
corrA = rA.corr()
off = corrA.values[~np.eye(len(corrA), dtype=bool)]
fig = px.imshow(corrA, color_continuous_scale="Viridis", zmin=0, zmax=1, labels=dict(color="Correlation"),
                title=f"Pairwise return correlation, 20 coins with full data from 2017-11-09 (average {off.mean():.2f})")
fig.update_layout(width=700, height=650)
fig.show()

from cryptograph.analysis.dependence import average_pairwise_corr
avgc = average_pairwise_corr(r20, window=60, min_obs=40)
after = pd.Series(False, index=avgc.index)
for d in events["date"]:
    after.loc[d: d + pd.Timedelta(days=30)] = True
fig = add_events(go.Figure(go.Scatter(x=avgc.index, y=avgc.values, mode="lines", line=dict(color="#ffd600", width=2),
                                      name="60-day average correlation")))
fig.update_layout(title="Average cross-asset correlation over time (20 coins, 60-day rolling)",
                  yaxis_title="Mean pairwise correlation", height=430)
fig.show()
print(f"🔗 Average correlation: {off.mean():.2f} (min {off.min():.2f}, max {off.max():.2f}); "
      f"30 days after a crisis: {avgc[after].mean():.2f} vs other days: {avgc[~after].mean():.2f}")

🔗 Average correlation: 0.57 (min 0.39, max 0.80); 30 days after a crisis: 0.61 vs other days: 0.54


*What this shows:* Every pair of coins is positively correlated (0.39–0.80, average 0.57): a large part of each coin's daily move is shared
with the others. Co-movement also changes over time and rises after crises (0.61 in the 30 days after an event vs 0.54
otherwise). A per-coin LSTM cannot use this shared information; a graph model can.

## 11. Dynamic Graph: Edges Over Time & Threshold Choice

Each day, two coins are joined by an edge when their last 30 days of returns have |correlation| ≥ 0.6 (plus each coin's
two strongest links). Edges are **undirected**; each coin pair is counted once below.

In [12]:
edge_series = pd.Series([e.size(1) // 2 for e in data["edge_indices"]], index=timestamps)
n_active = data["active_mask"].numpy().sum(axis=1)
fig = go.Figure([
    go.Scatter(x=edge_series.index, y=edge_series, mode="lines", name="Edges (coin pairs)", line=dict(color="#00e676", width=1.2)),
    go.Scatter(x=edge_series.index, y=edge_series.rolling(30).mean(), mode="lines", name="30-day average", line=dict(color="#ffea00", width=2)),
    go.Scatter(x=edge_series.index, y=n_active * (n_active - 1) / 2, mode="lines", name="All possible pairs", line=dict(color="#757575", dash="dot"))])
fig.update_layout(title="Dynamic cryptocurrency network: coin pairs with 30-day |correlation| ≥ 0.6",
                  yaxis_title="Edges", height=430, legend=dict(x=0.02, y=0.98))
fig.show()

from cryptograph.data.graphs.correlation import rolling_corr_mats
C, D = rolling_corr_mats(r20.reindex(timestamps).to_numpy(), 30, 20)
iu = ~np.eye(len(asset_names), dtype=bool)
dens = pd.DataFrame([{"threshold": t, "split": s, "share of pairs connected": np.nanmean(np.abs(C[day_split == s][:, iu]) >= t)}
                     for t in (0.3, 0.4, 0.5, 0.6, 0.7) for s in ("train", "val", "test")])
fig = px.line(dens, x="threshold", y="share of pairs connected", color="split", markers=True,
              title="How many coin pairs get an edge at each threshold (0.6 chosen)")
fig.add_vline(x=0.6, line_dash="dash", line_color="#ff5252")
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()
chance = 1 - stats.norm.cdf(np.arctanh(0.3) * np.sqrt(27))
found = {s: np.nanmean(D[day_split == s][:, iu] > 0.3) for s in ("train", "val", "test")}
print("Directed 'coin i yesterday → coin j today' edges (lag-1 corr > 0.3): found "
      + ", ".join(f"{s} {v:.1%}" for s, v in found.items()) + f"; expected by pure chance {chance:.1%}")

Directed 'coin i yesterday → coin j today' edges (lag-1 corr > 0.3): found train 1.6%, val 3.6%, test 4.3%; expected by pure chance 5.4%


*What this shows:* The first chart is the daily graph the graph models receive. The edge count changes every day and grows as new coins are
listed (grey line = all possible pairs), so the model must be **temporal**. The second chart shows why 0.6: at 0.3–0.4
almost every pair is connected in validation and test, which carries no structure. The printed line shows that
"coin i yesterday → coin j today" edges appear no more often than chance at daily resolution, so the TGN takes direction
from **which coin shocked first**. The hourly data in the next stage re-tests lead-lag at finer resolution.

## 12. Tensor & DataLoader Check

Shapes and sample counts for each model, loaded through the real DataLoaders, with a check for NaN/inf inputs.

In [13]:
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PygLoader
from cryptograph.data.datasets import LSTMWindowDataset, GraphSnapshotDataset, load_event_stream

rows = []
for s in ("train", "val", "test"):
    lstm = LSTMWindowDataset(f"../data/processed/v1/lstm_{s}.pt")
    b = next(iter(DataLoader(lstm, batch_size=64)))
    rows.append({"model": "LSTM windows", "split": s, "samples": len(lstm), "one batch": f"x {tuple(b['x'].shape)}, y {tuple(b['y_ret'].shape)}"})
    snap = GraphSnapshotDataset("../data/processed/v1/graph_snapshots.pt", s)
    g = next(iter(PygLoader(snap, batch_size=8)))
    rows.append({"model": "graph snapshots", "split": s, "samples": len(snap), "one batch": f"x {tuple(g.x.shape)}, edges {tuple(g.edge_index.shape)}"})
ev = load_event_stream("../data/processed/v1/tgn_events.pt")
for s in ("train", "val", "test"):
    rows.append({"model": "TGN events", "split": s, "samples": ev[s].num_events,
                 "one batch": f"src/dst/t {tuple(ev[s].src[:200].shape)}, msg {tuple(ev[s].msg[:200].shape)}"})
tensors = pd.DataFrame(rows)
display(tensors.pivot(index="model", columns="split", values="samples")[["train", "val", "test"]])
display(tensors[tensors["split"] == "train"][["model", "one batch"]].reset_index(drop=True))
print(f"✅ Inputs contain no NaN/inf: {bool(torch.isfinite(b['x']).all() and torch.isfinite(g.x).all() and torch.isfinite(ev['train'].msg).all())}; "
      f"event times sorted: {bool((ev['full'].t[1:] >= ev['full'].t[:-1]).all())}")

split,train,val,test
model,,,
LSTM windows,28880,9725,18820
TGN events,16266,8241,14877
graph snapshots,2043,489,941


,model,one batch
0,LSTM windows,"x (64, 30, 12), y (64, 5)"
1,graph snapshots,"x (160, 12), edges (2, 118)"
2,TGN events,"src/dst/t (200,), msg (200, 5)"


✅ Inputs contain no NaN/inf: True; event times sorted: True


*What this shows:* Each model's data loads through its real DataLoader. The LSTM gets 30-day windows of 12 features per coin; the graph
models get one graph per day (8 days × 20 coins = 160 nodes per batch); the TGN gets a time-sorted stream of shock
events with 5 message features. There are no NaN or infinite inputs. With about 2,000 training days the dataset is small
for deep learning, so models must be small, regularised (dropout, weight decay) and early-stopped on validation.

# Part II — Hourly Data (Binance, 30 coins)

The models now train on **hourly** candles from the Binance public archive (data.binance.vision, SHA-256 verified):
30 model coins from 2020-01-01 to 2026-09-27, plus LUNC and FTT as event coins. **1h** is the main resolution and
**2h** is the robustness check. Prices are in **USDT**, not USD. The split dates are the same as the daily data, with a
7-day embargo before validation and test.

13. **Coverage & quality.**
14. **Intraday seasonality.**
15. **Lead-lag** — does BTC or ETH move first?
16. **Epps effect & the graph threshold per resolution.**
17. **Hourly labels & the contagion jump.**
18. **Hourly tensor & DataLoader check.**

In [14]:
from plotly.subplots import make_subplots
from cryptograph.analysis.lead_lag import epps_curve, granger_leader_table, lead_lag_table, ccf_profile
from cryptograph.data.features.intraday import bucket_index
from cryptograph.data.graphs.correlation import pairwise_corr, threshold_for_density

V2 = "../data/processed/v2"
hp = torch.load(f"{V2}/1h/panel.pt", weights_only=False)
hmeta = {res: json.loads(Path(f"{V2}/{res}/metadata.json").read_text()) for res in ("1h", "2h")}
hq = pd.read_csv("../reports/data_quality_hourly.csv").set_index("coin")
hx = pd.read_csv("../reports/hourly_crosscheck.csv").set_index("coin")
continuity = pd.read_csv("../reports/binance_continuity.csv")
peg = pd.read_csv("../reports/usdt_peg.csv")

htimes = pd.to_datetime(hp["times"])
hcoins, hmodel = hp["coins"], hp["model_coins"]
hsplit = hp["split_id"].numpy()
SPLIT_NAMES = ["train", "val", "test"]
print(f"📊 {len(htimes):,} hours × {len(hcoins)} coins ({len(hmodel)} model + {len(hcoins) - len(hmodel)} event) "
      f"({htimes[0]} → {htimes[-1]}); {hmeta['1h']['shape']['F']} features at 1h, {hmeta['2h']['shape']['F']} at 2h")
print("🪙", ", ".join(hmodel))

📊 59,088 hours × 32 coins (30 model + 2 event) (2020-01-01 00:00:00 → 2026-09-27 23:00:00); 6 features at 1h, 7 at 2h


🪙 BTC, ETH, BNB, XRP, SOL, TRX, ZEC, DOGE, LINK, ADA, XLM, NEAR, BCH, UNI, LTC, AVAX, SUI, HBAR, QNT, SHIB, ENA, AAVE, DOT, WLD, PEPE, ICP, ETC, ARB, POL, JUP


*What this shows:* Loads the hourly pipeline outputs: 59,088 hours × 32 coins (30 model coins plus LUNC and FTT as event coins). After
removing redundant features on train, 6 features remain at 1h and 7 at 2h (at 1h the 7-day volatility duplicates the
3-day volatility, ρ = 0.90, so it is dropped; at 2h it is kept).

## 13. Hourly Coverage & Quality

Share of hours with a Binance candle per coin and month, then the quality numbers that matter for modelling.

In [15]:
act = pd.DataFrame(hp["active"].numpy(), index=htimes, columns=hcoins)
monthly = act.resample("MS").mean().T
order = act.idxmax().sort_values().index                  # by first active hour
fig = px.imshow(monthly.loc[order], aspect="auto", color_continuous_scale="Blues", zmin=0, zmax=1,
                labels=dict(color="share of hours"), title="Hourly coverage: share of hours with a candle (per month)")
for d, name in ((pd.Timestamp("2022-07-01"), "val"), (pd.Timestamp("2024-01-01"), "test")):
    fig.add_vline(x=d, line_dash="dot", line_color="white")
    fig.add_annotation(x=d, y=1.02, yref="paper", text=f"{name} starts", showarrow=False, font=dict(size=10))
fig.update_layout(height=700, xaxis_title=None, yaxis_title=None)
fig.show()

qt = pd.DataFrame({
    "first hour": pd.to_datetime(hq["first_hour"]).dt.date,
    "coverage": hq["window_coverage"].map("{:.1%}".format),
    "test coverage": hq["test_coverage"].map("{:.1%}".format),
    "gaps": hq["n_gaps"], "longest gap (h)": hq["longest_gap_hours"], "filled hours": hq["filled"],
    "vs daily: median diff": hx["median_rel_diff"].map(lambda v: f"{v:.2%}" if pd.notna(v) else "—"),
    "days > 1% off": hx["share_gt_1pct"].map(lambda v: f"{v:.1%}" if pd.notna(v) else "—"),
}).loc[[c for c in order if c in hq.index]]
display(qt)
print("Ticker continuity:")
display(continuity.dropna(axis=1, how="all"))
print(f"USDT off-peg days (> 0.5% from $1, Coin Metrics, checked until 2026-05-23): {', '.join(peg['date'])}")

,first hour,coverage,test coverage,gaps,longest gap (h),filled hours,vs daily: median diff,days > 1% off
coin,,,,,,,,
BTC,2020-01-01,100.0%,100.0%,15.0,5.0,13.0,0.06%,0.4%
POL,2020-01-01,99.8%,99.7%,16.0,79.0,13.0,0.10%,4.0%
ETC,2020-01-01,100.0%,100.0%,15.0,5.0,13.0,0.09%,1.0%
HBAR,2020-01-01,100.0%,100.0%,15.0,5.0,13.0,0.12%,4.3%
LTC,2020-01-01,100.0%,100.0%,15.0,5.0,13.0,0.09%,1.3%
BCH,2020-01-01,100.0%,100.0%,15.0,5.0,13.0,0.10%,1.4%
XLM,2020-01-01,100.0%,100.0%,15.0,5.0,13.0,0.10%,2.9%
ADA,2020-01-01,100.0%,100.0%,15.0,5.0,13.0,0.10%,1.1%
FTT,2020-01-01,87.3%,100.0%,15.0,7467.0,12.0,0.11%,1.6%


Ticker continuity:


,coin,type,from_symbol,to_symbol,old_last_hour,new_first_hour,gap_hours,log_price_change,flag,symbol,rows_dropped,first_dropped_hour,median_close_of_dropped_rows
0,POL,symbol_boundary,MATICUSDT,POLUSDT,2024-09-10 02:00:00,2024-09-13 10:00:00,79.0,0.000000,False,NaN,NaN,NaN,NaN
1,LUNC,reused_symbol_rows_dropped,NaN,NaN,NaN,NaN,NaN,NaN,False,LUNAUSDT,18.0,2022-05-31 06:00:00,8.84305
2,LUNC,symbol_boundary,LUNAUSDT,LUNCUSDT,2022-05-13 00:00:00,2022-09-09 08:00:00,2863.0,2.337373,False,NaN,NaN,NaN,NaN


USDT off-peg days (> 0.5% from $1, Coin Metrics, checked until 2026-05-23): 2020-03-12, 2020-03-13, 2023-03-11, 2023-03-12


*What this shows:* Coins enter when Binance lists them: 15 model coins have data from January 2020, while ARB, SUI, PEPE, WLD, JUP and
ENA start in 2023–24, so they have **no training data**. They still enter validation and test through the masks, with
the global train scaler. Every model coin covers at least 90% of the test period (lowest: ENA, 90.8%, listed April 2024).
The gaps are exchange-wide maintenance windows of at most 5 hours: gaps of up to 2 hours are filled and flagged (13 hours), longer
ones are masked. Two ticker changes are handled: MATIC→POL (79-hour gap, 0.0% price jump, a 1:1 swap) and LUNC
(LUNAUSDT until the May 2022 collapse, then LUNCUSDT from September 2022; the 2,863-hour gap stays masked and the new
LUNA is never spliced in). FTT's 7,467-hour gap is its delisting after FTX collapsed. Aggregated to days, the hourly
closes match the daily dataset to a median of about 0.1% (USDT vs USD and different closing times explain the rest).
SHIB is the exception (1.9%), because Yahoo rounds SHIB's ~$0.00001 price to 6 decimals, so it is the daily
reference that is off, not Binance. USDT left its $1 peg by more than 0.5% on four days (the March 2020 crash and the
March 2023 USDC/SVB scare); those days are flagged, not removed.

## 14. Intraday Seasonality

Average volatility and volume by weekday and hour (UTC), fitted on **train only** and pooled over coins after
scaling each coin to its own level. The pipeline divides these patterns out before computing features and shocks.

In [16]:
seas = json.loads(Path(f"{V2}/1h/seasonality.json").read_text())
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
vol = np.sqrt(np.array(seas["vol"]).reshape(7, 24))      # relative volatility (1 = average hour)
volu = np.exp(np.array(seas["logvol"]).reshape(7, 24))   # relative volume (1 = average hour)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Relative volatility", "Relative volume"), horizontal_spacing=0.08)
fig.add_trace(go.Heatmap(z=vol, x=list(range(24)), y=days, colorscale="Viridis", coloraxis="coloraxis",
                         hovertemplate="%{y} %{x}:00 UTC → ×%{z:.2f}<extra></extra>"), 1, 1)
fig.add_trace(go.Heatmap(z=volu, x=list(range(24)), y=days, colorscale="Viridis", coloraxis="coloraxis",
                         hovertemplate="%{y} %{x}:00 UTC → ×%{z:.2f}<extra></extra>"), 1, 2)
fig.update_layout(height=360, coloraxis=dict(colorscale="Viridis", colorbar_title="× average"),
                  title="Train-fitted seasonal profile (hour of day, UTC)")
fig.update_xaxes(title="hour (UTC)", dtick=3)
fig.update_yaxes(autorange="reversed")
fig.show()
peak = np.unravel_index(vol.argmax(), vol.shape)
low = np.unravel_index(vol.argmin(), vol.shape)
print(f"Most volatile hour: {days[peak[0]]} {peak[1]}:00 UTC (×{vol.max():.2f}); quietest: {days[low[0]]} {low[1]}:00 UTC (×{vol.min():.2f}); "
      f"weekend vs weekday volume: ×{volu[5:].mean() / volu[:5].mean():.2f}")

Most volatile hour: Wed 12:00 UTC (×2.17); quietest: Sun 6:00 UTC (×0.69); weekend vs weekday volume: ×0.77


*What this shows:* Volatility is strongly seasonal. The quietest hour (Sunday 06:00 UTC) has 0.69× the average volatility and the
busiest (Wednesday 12:00 UTC) has 2.17×. The busiest hours line up with the US morning (US economic data come out at
8:30 New York time). Weekend volume is 0.77× the weekday level. Without correcting for this, a normal Wednesday-afternoon move
would be labelled a shock and Sunday mornings would almost never produce one. The pipeline divides this profile out
(fitted on train only) before computing volatility features and shock labels.

## 15. Lead-Lag: Does BTC or ETH Move First?

If BTC or ETH moved first and other coins followed an hour later, the graph should carry directed lead-lag edges.
Three checks on **volatility-standardised returns** (return ÷ trailing 7-day volatility, so crash hours don't dominate),
using coins with at least 2,000 train hours:

- **Cross-correlation** corr(leader at t, coin at t+k) for k = −24…+24 hours. The band is a **circular-shift null**:
  the coin's series is rotated by a random offset (≥ 30 days), which keeps its own volatility clustering but breaks
  any timing link. The band is the 99th percentile of the largest |correlation| at any non-zero lag.
- **Granger tests** (3 lags), leader → coin and coin → leader, **Benjamini–Hochberg** corrected, with the extra R² as effect size.
- **Replication**: is the lag-1 pattern found on train the same on validation?

The rule for adding directed lead-lag edges to the graph and the TGN messages: only if the effect beats the null,
is asymmetric (leader → coin, not both ways), is large enough to matter (ΔR² ≥ 1%), and replicates on validation.

In [17]:
z = pd.DataFrame(hp["shock_z"].numpy(), index=htimes, columns=hcoins)[hmodel]
z_tr, z_va = z[hsplit == 0], z[hsplit == 1]
leaders = ["BTC", "ETH"]
prof, ll = lead_lag_table(z_tr, leaders, max_lag=24, n_null=500, seed=0)
gr = granger_leader_table(z_tr, leaders, lags=3)

med = prof.groupby(["leader", "lag"])["ccf"].median().reset_index()
fig = px.line(med[med["lag"] != 0], x="lag", y="ccf", color="leader", markers=True,
              title=f"Median cross-correlation, leader at t vs coin at t+k (train, {ll['coin'].nunique()} coins)")
q99 = ll["null_q99"].median()
fig.add_hrect(y0=-q99, y1=q99, fillcolor="grey", opacity=0.25, line_width=0, annotation_text="99% null band",
              annotation_position="top left")
for lead in leaders:
    c0 = med[(med["leader"] == lead) & (med["lag"] == 0)]["ccf"].item()
    fig.add_annotation(x=0, y=0.06 if lead == "BTC" else 0.052, text=f"lag 0 ({lead}): {c0:.2f} — off the scale",
                       showarrow=False, font=dict(size=11))
fig.update_layout(height=380, yaxis_range=[-0.07, 0.07], xaxis_title="lag k (hours; positive = leader first)",
                  yaxis_title="correlation")
fig.show()

ll_va = pd.DataFrame([{"leader": r.leader, "coin": r.coin,
                       "lag1_train": ccf_profile(z_tr[r.leader].to_numpy(), z_tr[r.coin].to_numpy(), 1)[2],
                       "lag1_val": ccf_profile(z_va[r.leader].to_numpy(), z_va[r.coin].to_numpy(), 1)[2]}
                      for r in ll.itertuples()])
summary = pd.DataFrame({
    "pairs": ll.groupby("leader").size(),
    "lag-0 corr (median)": ll.groupby("leader")["contemporaneous"].median().round(2),
    "largest |corr| at lag ≠ 0 (median)": ll.groupby("leader")["max_abs_nonzero"].median().round(3),
    "most common lag of that maximum (h)": ll.groupby("leader")["best_lag"].agg(lambda s: s.mode().iloc[0]),
    "pairs beating the 99% null": ll.groupby("leader")["exceeds_q99"].sum(),
    "lag-1 pattern train↔val (ρ across coins)": ll_va.groupby("leader").apply(
        lambda d: d["lag1_train"].corr(d["lag1_val"]), include_groups=False).round(2),
})
display(summary)
g = gr.groupby(["leader", "direction"]).agg(tests=("p", "size"), significant_BH5=("p_bh", lambda s: int((s < 0.05).sum())),
                                            median_dR2=("dR2", "median"), max_dR2=("dR2", "max"))
g["median_dR2"] = g["median_dR2"].map("{:.3%}".format)
g["max_dR2"] = g["max_dR2"].map("{:.3%}".format)
display(g)
top = gr.sort_values("dR2", ascending=False).head(3)
print("Largest Granger effects:", "; ".join(f"{r.cause}→{r.effect} ΔR² {r.dR2:.2%} (p_BH {r.p_bh:.1e})" for r in top.itertuples()))

,pairs,lag-0 corr (median),largest |corr| at lag ≠ 0 (median),most common lag of that maximum (h),pairs beating the 99% null,lag-1 pattern train↔val (ρ across coins)
leader,,,,,,
BTC,23,0.65,0.034,24,19,-0.11
ETH,22,0.68,0.035,24,20,-0.40


tests  significant_BH5 median_dR2 max_dR2
leader direction                                                
BTC    coin -> leader     23                3     0.030%  0.064%
       leader -> coin     23                5     0.022%  0.308%
ETH    coin -> leader     22                7     0.053%  0.137%
       leader -> coin     22                3     0.020%  0.473%

Largest Granger effects: ETH→QNT ΔR² 0.47% (p_BH 2.6e-06); BTC→QNT ΔR² 0.31% (p_BH 8.1e-04); DOT→ETH ΔR² 0.14% (p_BH 1.8e-03)


*What this shows:* Coins move **together within the same hour** (lag-0 correlation 0.65 with BTC, 0.68 with ETH), but not one after
another. At every non-zero lag the correlation stays tiny (median of the largest value: 0.035). It does exceed the
circular-shift null for most pairs, but almost all of those maxima sit at **+24 hours**: that is a daily cycle (BTC's own
return 24 hours ago is slightly negatively related to now), not one coin moving first. A shift-based null cannot tell
the two apart, so it is only an upper bound. The Granger tests, which control for each coin's own past, settle it:
after BH correction only 8 of 45 leader → coin tests and 10 of 45 coin → leader tests are significant, so neither
direction dominates and there is no "leader". The largest extra R² is 0.47% (ETH → QNT, a thinly traded coin whose
price updates late), and the median is 0.02%. The lag-1 pattern found on train does not repeat on validation
(ρ = −0.11 for BTC, −0.40 for ETH). **Decision:** the rule fails on asymmetry, size and replication, so no lead-lag
edge features are added and the hourly graphs carry no directed lag-1 edges. The directed networks for PageRank in
notebook 02 come from the GFEVD spillover table instead. The TGN keeps the trailing lag-1 correlation as one of its
5 message inputs, but we do not expect it to help.

## 16. Epps Effect & the Graph Threshold per Resolution

The **Epps effect**: correlations measured on short intervals are usually lower than on daily data, because prices
react with small delays. If it were strong, one threshold could not fit every resolution. First the average
correlation by sampling interval (train), then how dense the hourly graph is for each threshold.

In [18]:
lc = np.log(pd.read_parquet("../data/interim/hourly/master_1h.parquet")["close"].unstack("coin")
            .reindex(index=htimes, columns=hmodel))
lc_tr = lc[hsplit == 0]
lc_tr = lc_tr.loc[:, lc_tr.notna().sum() >= 2000]
ep = epps_curve(lc_tr, [1, 2, 4, 12, 24])
fig = px.line(ep, x="interval_hours", y="mean_corr", markers=True, log_x=True, text=ep["mean_corr"].map("{:.2f}".format),
              title=f"Epps effect: mean pairwise correlation by sampling interval (train, {lc_tr.shape[1]} coins)")
fig.update_traces(textposition="top center")
fig.update_layout(height=330, yaxis_range=[0.4, 0.75], xaxis=dict(tickvals=[1, 2, 4, 12, 24], title="interval (hours)"),
                  yaxis_title="mean correlation")
fig.show()

# density of the 1h graph for each threshold, per split, rebuilt exactly as the pipeline does
snaps = torch.load(f"{V2}/1h/graph_snapshots.pt", weights_only=False)
gi = snaps["graph_index"]
fvol = np.array(seas["vol"])[bucket_index(htimes, 1)]
r_ds = (np.log(pd.read_parquet("../data/interim/hourly/master_1h.parquet")["close"].unstack("coin")
               .reindex(index=htimes, columns=hcoins)).diff().to_numpy() / np.sqrt(fvol)[:, None])[:, gi]
act_g = hp["active"].numpy()[:, gi]
W = 168
st = snaps["snapshot_t"].numpy()
Cs = []
for t in st:
    C = pairwise_corr(r_ds[t - W + 1: t + 1], r_ds[t - W + 1: t + 1], int(0.8 * W))
    np.fill_diagonal(C, np.nan)
    Cs.append(C)
grid = np.round(np.arange(0.40, 0.91, 0.05), 2)
rows = []
for i, s in enumerate(SPLIT_NAMES):
    ks = [k for k, t in enumerate(st) if hsplit[t] == i]
    for th in grid:
        _, d = threshold_for_density([Cs[k] for k in ks], [act_g[st[k]] for k in ks], 1.0, grid=[th])
        rows.append({"split": s, "threshold": th, "density": d})
dens = pd.DataFrame(rows)
thr_tr, d_tr = threshold_for_density([Cs[k] for k, t in enumerate(st) if hsplit[t] == 0],
                                     [act_g[t] for t in st if hsplit[t] == 0], 0.30)
fig = px.line(dens, x="threshold", y="density", color="split", markers=True,
              title="1h graph: mean edge density vs |correlation| threshold (168-hour windows)")
fig.add_hline(y=0.30, line_dash="dot", annotation_text="daily graph: 30% on train")
fig.add_vline(x=thr_tr, line_dash="dash", line_color="#ff5252", annotation_text=f"fitted on train: {thr_tr}")
fig.add_vline(x=0.5, line_dash="dot", line_color="grey", annotation_text="old draft 0.5", annotation_position="bottom right")
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()
display(dens.pivot(index="threshold", columns="split", values="density")[SPLIT_NAMES].loc[[0.5, 0.75]].map("{:.0%}".format))
print(f"Recomputed threshold {thr_tr} (train density {d_tr:.1%}) = pipeline's {snaps['undirected_threshold']}: "
      f"{thr_tr == snaps['undirected_threshold']}; 2h threshold: {hmeta['2h']['graph_undirected_threshold']}")
display(pd.DataFrame({res: {**{f"density {s}": f"{v:.0%}" for s, v in m["graph_density_mean"]["undirected_by_split"].items()},
                            "threshold": m["graph_undirected_threshold"]} for res, m in hmeta.items()}).T)

split,train,val,test
threshold,,,
0.50,75%,79%,78%
0.75,29%,24%,35%


Recomputed threshold 0.75 (train density 29.5%) = pipeline's 0.75: True; 2h threshold: 0.77


,density train,density val,density test,threshold
1h,38%,30%,39%,0.75
2h,38%,29%,36%,0.77


*What this shows:* The Epps effect is **weak** in crypto: average correlation is 0.58 at 1h, 0.61–0.62 at 4–12h and 0.59 at 1 day.
Coins trade 24/7 on liquid exchanges, so prices catch up with each other within the hour. The hourly graph is dense for
a different reason: with 168-hour windows almost every pair clears 0.5, so the old draft threshold gave a 75% dense,
nearly complete graph that a GNN cannot learn much from. The threshold is now **fitted on train for each resolution** to
match the daily graph's 30% density: **0.75 at 1h and 0.77 at 2h**. Recomputing it here gives the same value as the
pipeline. With threshold edges alone, density is 29% on train, 24% on validation and 35% on test. The final graphs
also keep each coin's 2 strongest links, so no coin is isolated, which brings density to 30–39% (table).

## 17. Hourly Labels & the Contagion Jump

The shock label is |z| above a threshold chosen on train for a ~4% rate. Contagion: after coin *i* shocks, how often
does coin *j* shock within *k* hours, compared with the chance that a coin shocks in any *k*-hour window?

In [19]:
hb = pd.concat([pd.read_csv(f"../reports/hourly_{res}_label_balance.csv").assign(resolution=res) for res in ("1h", "2h")])
sh = hb[hb["label"].str.startswith("shock_next")]
display(sh.pivot(index="resolution", columns="split", values="rate")[SPLIT_NAMES].map("{:.1%}".format)
        .assign(threshold=sh.groupby("resolution")["label"].first().str.extract(r"> ?([\d.]+)")[0]))

ct = hb[hb["label"].str.startswith("contagion")].copy()
ct["k (hours)"] = ct["label"].str.extract(r"k(\d+)h")[0].astype(int)
ct["lift"] = ct["rate"] / ct["base_rate_any_coin"]
tr = ct[ct["split"] == "train"]
long = pd.concat([tr.assign(case="after coin i shocks", value=tr["rate"]),
                  tr.assign(case="base rate (any window)", value=tr["base_rate_any_coin"])])
fig = px.bar(long, x="k (hours)", y="value", color="case", barmode="group", facet_col="resolution",
             text=[f"{v:.0%}" for v in long["value"]], title="Contagion (train): chance coin j shocks within k hours")
fig.update_xaxes(type="category")
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()
display(ct.pivot_table(index=["resolution", "k (hours)"], columns="split", values="lift")[SPLIT_NAMES].round(2)
        .rename(columns=lambda c: f"lift {c}"))

split,train,val,test,threshold
resolution,,,,
1h,4.0%,3.6%,3.9%,2.29
2h,4.0%,3.8%,4.0%,2.33


split                 lift train  lift val  lift test
resolution k (hours)                                 
1h         1                2.33      2.37       2.71
           3                1.94      1.78       2.01
           6                1.68      1.52       1.70
           12               1.44      1.29       1.44
           24               1.26      1.16       1.27
2h         1                2.14      1.78       2.29
           3                1.88      1.49       1.93
           6                1.74      1.43       1.77
           12               1.55      1.25       1.50
           24               1.37      1.23       1.33

*What this shows:* The shock label is positive in about 4% of hours in every split, at both resolutions (thresholds 2.29 and 2.33,
both set on train), so class weights or focal loss and AUPRC remain the right choices. The contagion signal is much clearer
than at daily resolution: in the hour after coin *i* shocks, coin *j* shocks **2.3× more often** than in a random
hour (train: 9.3% vs 4.0%). The lift is the same on validation (2.4×) and test (2.7×), so it is not a train-period
artefact. It fades with the horizon (1.3× within 24 hours), so contagion mostly happens within the first few hours,
which is why the models work at 1h. At 2h, "within 1 hour" means the next 2-hour bar, and the lift there is 2.1×.

## 18. Hourly Tensor & DataLoader Check

Each hourly dataset loaded through its real loader: LSTM windows sliced on the fly from the shared panel, hourly
graph snapshots as PyG graphs, and the shock-triggered TGN event stream.

In [20]:
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PygLoader
from cryptograph.data.datasets import HourlySnapshotDataset, PanelWindowDataset, load_event_stream

rows, finite = [], True
for res in ("1h", "2h"):
    for s in SPLIT_NAMES:
        ds = PanelWindowDataset(f"{V2}/{res}/panel.pt", f"{V2}/{res}/lstm_index_{s}.pt")
        b = next(iter(DataLoader(ds, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(0))))
        finite &= bool(torch.isfinite(b["x"]).all())
        rows.append({"res": res, "dataset": "LSTM windows", "split": s, "samples": len(ds),
                     "one batch": f"x {tuple(b['x'].shape)}, y_ret {tuple(b['y_ret'].shape)}"})
        gs = HourlySnapshotDataset(f"{V2}/{res}/panel.pt", f"{V2}/{res}/graph_snapshots.pt", s)
        g = next(iter(PygLoader(gs, batch_size=8)))
        finite &= bool(torch.isfinite(g.x).all())
        rows.append({"res": res, "dataset": "graph snapshots", "split": s, "samples": len(gs),
                     "one batch": f"x {tuple(g.x.shape)}, edges {tuple(g.edge_index.shape)}"})
    ev = load_event_stream(f"{V2}/{res}/tgn_events.pt")
    for s in SPLIT_NAMES:
        finite &= bool(torch.isfinite(ev[s].msg).all())
        rows.append({"res": res, "dataset": "TGN events", "split": s, "samples": ev[s].num_events,
                     "one batch": f"msg {tuple(ev[s].msg[:200].shape)}, y {tuple(ev[s].y[:200].shape)}"})
    sorted_t = bool((ev["full"].t[1:] >= ev["full"].t[:-1]).all())
    del ev
ht = pd.DataFrame(rows)
display(ht.pivot_table(index=["res", "dataset"], columns="split", values="samples", sort=False)[SPLIT_NAMES])
display(ht[(ht["split"] == "train")][["res", "dataset", "one batch"]].reset_index(drop=True))
print(f"✅ Inputs contain no NaN/inf: {finite}; TGN event times sorted: {sorted_t}")

split                   train       val      test
res dataset                                      
1h  LSTM windows     427130.0  333094.0  711740.0
    graph snapshots     905.0     541.0     993.0
    TGN events       360590.0  300998.0  801559.0
2h  LSTM windows     213586.0  166548.0  355871.0
    graph snapshots     905.0     541.0     993.0
    TGN events       180209.0  158762.0  413362.0

,res,dataset,one batch
0,1h,LSTM windows,"x (64, 168, 6), y_ret (64, 3)"
1,1h,graph snapshots,"x (240, 6), edges (2, 408)"
2,1h,TGN events,"msg (200, 5), y (200, 5)"
3,2h,LSTM windows,"x (64, 84, 7), y_ret (64, 3)"
4,2h,graph snapshots,"x (240, 7), edges (2, 460)"
5,2h,TGN events,"msg (200, 5), y (200, 5)"


✅ Inputs contain no NaN/inf: True; TGN event times sorted: True


*What this shows:* Every hourly dataset loads through its real loader. At 1h: 427k / 333k / 712k LSTM windows (168 hours × 6
features each), 905 / 541 / 993 daily graph snapshots of the 30 model coins (a batch of 8 graphs = 240 nodes), and
360,590 / 300,998 / 801,559 TGN events (train / val / test), each with 5 message features and 5 contagion labels
(one per k). Inputs are finite and events are in time order. The test split is the largest because it is the
longest (2.7 years) and has the most listed coins. The full 1h TGN stream (1.46M events) is meant for the Colab GPU,
and the 2h stream (752k events) is the robustness check.

## Key Takeaways

**Part I — daily**
- **Clean and audited:** 90,643 coin-days, 28 bad prices corrected, no invented data; the early years (2017–18) are the least reliable.
- **Model on returns and volatility:** prices are non-stationary, returns are stationary but fat-tailed (kurtosis ≈ 18) → Huber loss, clipped inputs.
- **Targets:** volatility and shocks are predictable, direction is not; shocks are rare (~4%) → weighted/focal loss, AUPRC.
- **No leakage:** fixed-date splits with an embargo, scalers fitted on train only, labels built from past volatility.
- **Graph justified:** coins share most of their movement (correlation 0.57) and the network changes daily → temporal graph model.

**Part II — hourly (what the models train on)**
- **30 coins, ~1.6M coin-hours** from Binance, checksum-verified and within ~0.1% of the daily data; ticker changes and delistings masked, never spliced.
- **Strong intraday seasonality** (0.69× to 2.17× average volatility), removed with a train-fitted profile before features and shocks.
- **No usable lead-lag:** coins move together within the hour; no directed lag edges or lead-lag features.
- **Graph threshold fitted on train per resolution** (0.75 at 1h, 0.77 at 2h) for the same 30% density as the daily graph; the Epps effect is weak.
- **Contagion is fast:** a shock makes another coin's shock 2.3–2.7× more likely in the next hour, stable across splits, fading by 24 hours.

**Next:** `02_network_analysis.ipynb` (centrality, communities, MST, the Diebold–Yilmaz spillover network and the
independent-cascade baseline); the corrected daily Diebold–Yilmaz analysis is its section 2b.